# Sacred Fruit: reentrenamiento del modelo

Entorno de ejecución: **GPU T4**. Ejecutá todo en orden y pegá tu Private API Key de Roboflow cuando la pida.

- Usa la última versión de tu dataset de Roboflow (v3: suma los tomates y bananas pasadas que cargó tu compañero) y los datasets públicos de `EXTRA_DATASETS`, cada uno en su versión con menos aumentaciones de color.
- Las fotos sin etiquetas de tu dataset se descartan: si entraran, el modelo aprendería que esa fruta es fondo.
- Las bananas del dataset grande solo traen cajas. SAM las convierte a máscaras antes de entrenar.
- `DEFECT_DATASETS` suma fotos con manchas, golpes y podredumbre ya marcadas, para que el modelo aprenda la clase `defecto`. Esas fotos traen la fruta sin estado de madurez: lo pone el modelo anterior (`PSEUDO_MODEL`) y, si no está seguro, la foto se descarta.
- Si `PSEUDO_MODEL` está en Drive, el entrenamiento parte de ese modelo (ajuste fino, unas 30 épocas). Si no, entrena desde cero.
- Las copias de `best.pt`, `last.pt` y los gráficos van a Google Drive (`MyDrive/sacred_fruit/<RUN_NAME>`) cada 3 épocas. Si la sesión se corta, volvé a correr todo: sigue desde el último `last.pt`.

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics roboflow

In [ ]:
import getpass
ROBOFLOW_API_KEY = globals().get("ROBOFLOW_API_KEY") or getpass.getpass("Pegá tu Private API Key de Roboflow y apretá Enter: ")
WORKSPACE = "dario-avalos-reckziegel"
PROJECT = "sacred-fruit"
VERSION = None
RUN_NAME = "v4"

# (workspace, proyecto, mapeo de clases). Una foto con alguna etiqueta fuera del mapeo se descarta entera.
EXTRA_DATASETS = [
    ("yashas", "orange_ripeness", {"ripe_orange": "naranja_madura", "unripe_orange": "naranja_verde"}),
    ("yashas", "mango_ripeness", {"ripe_mango": "mango_maduro", "raw_mango": "mango_verde"}),
    ("apu-assignment", "tomato-ripeness-segmentation", {"fully_ripe": "tomate_maduro", "turning": "tomate_verde", "unripe": "tomate_verde"}),
    ("sign-l", "orange-kbdwo", {"partially_ripe_orange": "naranja_verde", "ripe_orange": "naranja_madura",
                                "rotten_orange": "naranja_sobrepasada", "unripe_orange": "naranja_verde"}),
    ("recognition-of-fruits", "fruit-grading-zrkn9", {"fresh_banana": "banana_madura", "fresh_orange": "naranja_madura",
                                                      "rotten_banana": "banana_sobrepasada", "rotten_orange": "naranja_sobrepasada"}),
    ("universitas-hasanuddin-lq2k8", "tomato-defects-80syu", {"ripe": "tomate_maduro", "half_ripe": "tomate_verde",
                                                               "raw": "tomate_verde", "rotten": "tomate_sobrepasado"}),
    ("sansan-5p4nm", "fruit-seg1", {"earlyripe_mango": "mango_verde", "partiallyripe_mango": "mango_maduro", "ripe_mango": "mango_maduro",
                                    "rotten_mango": "mango_sobrepasado", "unripe_mango": "mango_verde"}),
]
# Datasets con cajas (detección). SAM convierte cada caja en máscara. El número es el máximo de fotos a usar.
BOX_DATASETS = [
    ("fruit-ripening", "banana-ripening-process", {"unripe": "banana_verde", "freshunripe": "banana_verde", "ripe": "banana_madura",
                                                   "freshripe": "banana_madura", "overripe": "banana_sobrepasada", "rotten": "banana_sobrepasada"}, 1800),
]
# Datasets con defectos marcados. "defecto" es la zona dañada; el resto es la fruta sin estado.
DEFECT_DATASETS = [
    ("mcf-wascl", "fruit-ptiqf", {"orange": "naranja", "tomato": "tomate", "defect": "defecto"}),
    ("h2sagings-workspace", "banana-uuiln", {"banana": "banana", "bruise": "defecto"}),
    ("fruit-detection-fyqgb", "fruit-disease-segmentation-anlg0", {"mango": "mango", "healthy_mango": "mango",
                                                                   "alternaria_mango": "defecto", "rot_mango": "defecto"}),
]
PSEUDO_MODEL = "/content/drive/MyDrive/sacred_fruit/v3/best.pt"
BASE_MODEL = "yolo11s-seg.pt"
SAM_MODEL = "sam2.1_b.pt"
EPOCHS = None
IMGSZ = 640
BG_RATIO = 0.15
MAX_PER_CLASS = 1500
DRIVE_DIR = f"/content/drive/MyDrive/sacred_fruit/{RUN_NAME}"

In [ ]:
import os
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Sin Google Drive, las copias quedan solo en /content:', e)
    DRIVE_DIR = '/content/copias'
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
print('Copias en', DRIVE_DIR)

In [ ]:
import re, shutil, random, glob, json, zipfile, urllib.request
from collections import Counter, defaultdict
from roboflow import Roboflow

random.seed(42)
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
versions = sorted(int(str(v.version).split('/')[-1]) for v in project.versions())
ver = VERSION or versions[-1]
print("Versiones de tu dataset:", versions, "-> uso", ver)
ds = project.version(ver).download("yolov8", location="/content/rf_main", overwrite=True)

# De cada dataset público se baja la versión con menos aumentaciones (la original si existe):
# las aumentaciones de Roboflow cambian el color, y la madurez se decide justamente por el color.
def cleanest(ws, pj):
    p = rf.workspace(ws).project(pj)
    vid = lambda v: int(str(v.get('id', v.get('version', ''))).split('/')[-1])
    try:
        info = p.get_version_information()
        best = min(info, key=lambda v: (bool(v.get('augmentation')), v.get('images') or 10**9, -vid(v)))
        n = vid(best)
        print(f"{pj}: versión {n} | imágenes {best.get('images')} | aumentaciones {list((best.get('augmentation') or {}).keys()) or 'ninguna'}")
    except Exception as e:
        n = max(int(str(x.version).split('/')[-1]) for x in p.versions())
        print(f"{pj}: no pude leer las versiones ({e}), uso la {n}")
    return p.version(n), n

sources = [(Path(ds.location), None)]
for ws, pj, mapping in EXTRA_DATASETS:
    try:
        v, n = cleanest(ws, pj)
        d = v.download("yolov8", location=f"/content/rf_{pj}_v{n}", overwrite=False)
        sources.append((Path(d.location), mapping))
    except Exception as e:
        print("No se pudo bajar", pj, "->", e)
defect_sources = []
for ws, pj, mapping in DEFECT_DATASETS:
    try:
        v, n = cleanest(ws, pj)
        d = v.download("yolov8", location=f"/content/rf_{pj}_v{n}", overwrite=False)
        defect_sources.append((Path(d.location), mapping))
    except Exception as e:
        print("No se pudo bajar", pj, "->", e)
box_sources = []
for ws, pj, mapping, limit in BOX_DATASETS:
    try:
        v, n = cleanest(ws, pj)
        d = v.download("yolov8", location=f"/content/rf_{pj}_v{n}", overwrite=False)
        box_sources.append((Path(d.location), mapping, limit))
    except Exception as e:
        print("No se pudo bajar", pj, "->", e)

In [ ]:
import yaml
CLASSES = ['banana_madura','banana_sobrepasada','banana_verde','defecto','mango_maduro','mango_sobrepasado','mango_verde',
           'naranja_madura','naranja_sobrepasada','naranja_verde','tomate_maduro','tomate_sobrepasado','tomate_verde']
IDX = {c:i for i,c in enumerate(CLASSES)}
key = lambda n: re.sub(r'[\s\-]+', '_', str(n).strip().lower())

def norm(name):
    n = key(name)
    n = n.replace('maduro','madur_').replace('madura','madur_').replace('sobrepasado','sobrepasad_').replace('sobrepasada','sobrepasad_')
    for c in CLASSES:
        k = c.replace('maduro','madur_').replace('madura','madur_').replace('sobrepasado','sobrepasad_').replace('sobrepasada','sobrepasad_')
        if n == k: return c
    return None

def read_names(src):
    names = yaml.safe_load(open(src/'data.yaml'))['names']
    return names if isinstance(names, list) else [names[k] for k in sorted(names)]

def group_of(src, img):
    stem = re.split(r'_(jpg|jpeg|png|webp)\.rf\.', Path(img).name, flags=re.I)[0]
    return f"{src.name}/{stem}"

items, report = [], []
for src, mapping in sources:
    names = read_names(src)
    remap = {i: (mapping.get(key(n)) if mapping else norm(n)) for i, n in enumerate(names)}
    used = Counter(); dropped = Counter(); kept_imgs = 0
    for img in sorted(glob.glob(str(src/'*'/'images'/'*'))):
        lbl = Path(img.replace('/images/','/labels/')).with_suffix('.txt')
        lines, unknown = [], False
        if lbl.exists():
            for ln in open(lbl).read().splitlines():
                p = ln.split()
                if not p: continue
                c = remap.get(int(p[0]))
                if c is None:
                    unknown = True; dropped[names[int(p[0])]] += 1
                elif len(p) >= 7:
                    lines.append(' '.join([str(IDX[c])] + p[1:]))
        if unknown or not lines:
            continue
        kept_imgs += 1
        used.update(CLASSES[int(l.split()[0])] for l in lines)
        items.append(dict(img=img, lines=lines, group=group_of(src, img), src=src.name))
    report.append((src.name, kept_imgs, dict(used), dict(dropped)))
    print(f"{src.name}: {kept_imgs} fotos | clases originales {names}")
    print("   usadas:", dict(used))
    if dropped: print("   fotos descartadas por estas clases:", dict(dropped))
    if mapping and not used:
        print("   ATENCIÓN: ninguna clase coincidió con el mapeo, revisá los nombres de arriba")

In [ ]:
# Bananas con cajas -> máscaras con SAM. Se toma el contorno más grande de cada máscara.
import cv2, numpy as np, torch
from ultralytics import SAM

def mask_to_poly(mask):
    m = (mask > 0.5).astype(np.uint8)
    cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cs: return None
    c = max(cs, key=cv2.contourArea)
    if cv2.contourArea(c) < 50: return None
    c = cv2.approxPolyDP(c, 0.003 * cv2.arcLength(c, True), True).reshape(-1, 2)
    if len(c) < 3: return None
    h, w = m.shape
    return ' '.join(f"{x/w:.5f} {y/h:.5f}" for x, y in c)

def to_xyxy(r):
    if len(r) == 5:
        xc, yc, bw, bh = map(float, r[1:5]); return xc - bw/2, yc - bh/2, xc + bw/2, yc + bh/2
    pts = np.array(list(map(float, r[1:]))).reshape(-1, 2); (x1, y1), (x2, y2) = pts.min(0), pts.max(0)
    return x1, y1, x2, y2

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy; u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / u if u > 0 else 0

# Una misma banana con dos cajas de clases distintas es una etiqueta contradictoria: esa foto se descarta.
def conflicting(rows, remap):
    bx = [(remap[int(r[0])], to_xyxy(r)) for r in rows]
    return any(c1 != c2 and iou(b1, b2) > 0.5 for i, (c1, b1) in enumerate(bx) for c2, b2 in bx[i+1:])

if box_sources:
    sam = SAM(SAM_MODEL)
for src, mapping, limit in box_sources:
    names = read_names(src)
    remap = {i: mapping.get(key(n)) for i, n in enumerate(names)}
    out = Path(f'/content/sam_{src.name}'); shutil.rmtree(out, ignore_errors=True); (out/'images').mkdir(parents=True, exist_ok=True); (out/'labels').mkdir(parents=True, exist_ok=True)
    cand = []
    for img in sorted(glob.glob(str(src/'*'/'images'/'*'))):
        lbl = Path(img.replace('/images/','/labels/')).with_suffix('.txt')
        if not lbl.exists(): continue
        rows = [ln.split() for ln in open(lbl).read().splitlines() if ln.strip()]
        if rows and all(remap.get(int(r[0])) for r in rows) and not conflicting(rows, remap):
            cand.append((img, rows))
    by_group = defaultdict(list)
    for img, rows in cand: by_group[group_of(src, img)].append((img, rows))
    picked = [v[0] for v in by_group.values()]
    random.shuffle(picked); picked = picked[:limit]
    used = Counter()
    for j, (img, rows) in enumerate(picked):
        im = cv2.imread(img); h, w = im.shape[:2]
        boxes, cls = [], []
        for r in rows:
            x1, y1, x2, y2 = to_xyxy(r)
            boxes.append([x1*w, y1*h, x2*w, y2*h]); cls.append(remap[int(r[0])])
        res = sam(img, bboxes=boxes, verbose=False)[0]
        lines = []
        if res.masks is not None:
            for k, mk in enumerate(res.masks.data.cpu().numpy()[:len(cls)]):
                poly = mask_to_poly(mk)
                if poly: lines.append(f"{IDX[cls[k]]} {poly}")
        if len(lines) == len(cls):
            dst = out/'images'/Path(img).name
            shutil.copy(img, dst); open(out/'labels'/(dst.stem + '.txt'), 'w').write('\n'.join(lines))
            items.append(dict(img=str(dst), lines=lines, group=f"sam_{src.name}/{Path(img).stem}", src=f"sam_{src.name}"))
            used.update(cls)
        if j % 200 == 0: print(f"SAM {j}/{len(picked)}"); torch.cuda.empty_cache()
    print(f"{src.name}: {sum(1 for it in items if it['src']==f'sam_{src.name}')} fotos convertidas | {dict(used)}")
if box_sources:
    del sam; torch.cuda.empty_cache()

In [ ]:
# Fotos con defectos: el estado de cada fruta lo pone PSEUDO_MODEL. Cada fruta tiene que coincidir con una
# detección del mismo tipo (IoU 0.5 o más); si alguna no coincide, la foto se descarta para no meter estados dudosos.
from ultralytics import YOLO
teacher = YOLO(PSEUDO_MODEL) if defect_sources and Path(PSEUDO_MODEL).exists() else None
if defect_sources and teacher is None:
    print("No encontré", PSEUDO_MODEL, "-> no se usan los datasets de defectos")
for src, mapping in (defect_sources if teacher else []):
    names = read_names(src)
    remap = {i: mapping.get(key(n)) for i, n in enumerate(names)}
    stats, used = Counter(), Counter()
    for j, img in enumerate(sorted(glob.glob(str(src/'*'/'images'/'*')))):
        lbl = Path(img.replace('/images/','/labels/')).with_suffix('.txt')
        rows = [ln.split() for ln in open(lbl).read().splitlines() if ln.strip()] if lbl.exists() else []
        if not rows or any(remap.get(int(r[0])) is None or len(r) < 7 for r in rows):
            stats['otras clases o sin polígono'] += 1; continue
        fruits = [r for r in rows if remap[int(r[0])] != 'defecto']
        defects = [r for r in rows if remap[int(r[0])] == 'defecto']
        if not fruits or not defects:
            stats['sin fruta o sin defecto'] += 1; continue
        pred = teacher.predict(img, conf=0.3, imgsz=IMGSZ, verbose=False)[0]
        dets = [(pred.names[int(c)], b) for c, b in zip(pred.boxes.cls.tolist(), pred.boxes.xyxyn.tolist())]
        lines = []
        for r in fruits:
            f, box = remap[int(r[0])], to_xyxy(r)
            score, label = max(((iou(box, b), n) for n, b in dets if n.startswith(f + '_')), default=(0, None))
            if score < 0.5: break
            lines.append(' '.join([str(IDX[label])] + r[1:]))
        else:
            lines += [' '.join([str(IDX['defecto'])] + r[1:]) for r in defects]
            items.append(dict(img=img, lines=lines, group=group_of(src, img), src=src.name))
            used.update(CLASSES[int(l.split()[0])] for l in lines); stats['usadas'] += 1
            continue
        stats['estado dudoso'] += 1
        if j % 200 == 0: torch.cuda.empty_cache()
    print(f"{src.name}: {dict(stats)}")
    print("   etiquetas:", dict(used))
if teacher is not None:
    del teacher; torch.cuda.empty_cache()

In [ ]:
# Revisión visual: 3 fotos por cada origen con sus polígonos y clases. Si algo se ve mal, se saca ese dataset y se vuelve a correr.
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
COL = ['#e6194b','#3cb44b','#ffe119','#4363d8','#f58231','#911eb4','#46f0f0','#f032e6','#bcf60c','#fabebe','#008080','#e6beff','#9a6324']
by_src = defaultdict(list)
for it in items: by_src[it['src']].append(it)
for s, its in by_src.items():
    sample = random.sample(its, min(3, len(its)))
    fig, axes = plt.subplots(1, len(sample), figsize=(5*len(sample), 5))
    for ax, it in zip(np.atleast_1d(axes), sample):
        im = Image.open(it['img']).convert('RGB'); w, h = im.size; d = ImageDraw.Draw(im, 'RGBA')
        for ln in it['lines']:
            p = ln.split(); c = int(p[0]); pts = [(float(x)*w, float(y)*h) for x, y in zip(p[1::2], p[2::2])]
            d.polygon(pts, outline=COL[c], fill=COL[c] + '55', width=3); d.text(pts[0], CLASSES[c], fill='white')
        ax.imshow(im); ax.axis('off')
    plt.suptitle(f"{s} ({len(its)} fotos)"); plt.tight_layout(); plt.savefig(f"{DRIVE_DIR}/muestra_{s}.jpg"); plt.show()

groups = defaultdict(list)
for it in items: groups[it['group']].append(it)
print(f"{len(items)} imágenes, {len(groups)} fotos originales")
cnt = Counter(CLASSES[int(l.split()[0])] for it in items for l in it['lines'])
for c in CLASSES: print(f"  {c:22s} {cnt[c]}")

In [ ]:
# Submuestreo: se descartan fotos originales enteras solo si todas sus clases ya pasaron MAX_PER_CLASS.
# Las fotos de tu dataset van primero para que nunca se descarten.
keys = sorted(groups); random.shuffle(keys)
keys.sort(key=lambda k: not k.startswith('rf_main/'))
kept, used = {}, Counter()
for k in keys:
    its = groups[k]
    cls = Counter(CLASSES[int(l.split()[0])] for it in its for l in it['lines'])
    if cls and all(used[c] >= MAX_PER_CLASS for c in cls):
        continue
    kept[k] = its; used.update(cls)
print(f"Fotos originales: {len(groups)} -> {len(kept)}")
for c in CLASSES: print(f"  {c:22s} {cnt[c]:6d} -> {used[c]}")
groups = kept

In [ ]:
keys = sorted(groups); random.Random(7).shuffle(keys)
nval = max(1, int(len(keys)*0.15))
split = {k: ('val' if i < nval else 'train') for i, k in enumerate(keys)}

OUT = Path('/content/data'); shutil.rmtree(OUT, ignore_errors=True)
for s in ['train','val','neg_test']:
    (OUT/s/'images').mkdir(parents=True); (OUT/s/'labels').mkdir(parents=True)

def put(it, s, i):
    ext = Path(it['img']).suffix
    shutil.copy(it['img'], OUT/s/'images'/f"{i:05d}{ext}")
    open(OUT/s/'labels'/f"{i:05d}.txt",'w').write('\n'.join(it['lines']))

i = 0
for k in keys:
    its = groups[k]
    for it in (its if split[k]=='train' else its[:1]):
        put(it, split[k], i); i += 1
ntrain = len(os.listdir(OUT/'train'/'images'))
print('train', ntrain, 'val', len(os.listdir(OUT/'val'/'images')))

In [ ]:
# Fondos sin fruta de COCO val2017. Se excluye toda foto con comida, mesas, bowls o heladeras: ahí aparecen
# tomates y naranjas sin etiquetar, y el modelo aprendería que esa fruta es fondo. Las manzanas y peras las rechaza CLIP en el backend.
if not Path('/content/coco/val2017').exists():
    os.makedirs('/content/coco', exist_ok=True)
    urllib.request.urlretrieve('http://images.cocodataset.org/zips/val2017.zip', '/content/coco/val.zip')
    urllib.request.urlretrieve('http://images.cocodataset.org/annotations/annotations_trainval2017.zip', '/content/coco/ann.zip')
    zipfile.ZipFile('/content/coco/val.zip').extractall('/content/coco')
    zipfile.ZipFile('/content/coco/ann.zip').extractall('/content/coco')
ann = json.load(open('/content/coco/annotations/instances_val2017.json'))
cat = {c['id']: c for c in ann['categories']}
bad = {a['image_id'] for a in ann['annotations']
       if cat[a['category_id']]['supercategory'] == 'food' or cat[a['category_id']]['name'] in ('dining table', 'bowl', 'refrigerator')}
bg = sorted(im['file_name'] for im in ann['images'] if im['id'] not in bad)
random.Random(11).shuffle(bg)
n_bg = int(ntrain * BG_RATIO / (1 - BG_RATIO))
for j, f in enumerate(bg[:n_bg]):
    shutil.copy(f'/content/coco/val2017/{f}', OUT/'train'/'images'/f'bg_{j:05d}.jpg')
    open(OUT/'train'/'labels'/f'bg_{j:05d}.txt','w').close()
for j, f in enumerate(bg[n_bg:n_bg+300]):
    shutil.copy(f'/content/coco/val2017/{f}', OUT/'neg_test'/'images'/f'neg_{j:05d}.jpg')
print('fondos agregados al train:', n_bg, '| negativos de prueba: 300')

yaml.safe_dump({'path': str(OUT), 'train': 'train/images', 'val': 'val/images', 'names': {i:c for i,c in enumerate(CLASSES)}},
               open(OUT/'data.yaml','w'), allow_unicode=True, sort_keys=False)

In [ ]:
from ultralytics import YOLO

INIT = PSEUDO_MODEL if Path(PSEUDO_MODEL).exists() else None
if EPOCHS is None:
    EPOCHS = 30 if INIT else 100 if ntrain < 3000 else 80 if ntrain < 6000 else 60
print('Parto de:', INIT or BASE_MODEL, '| épocas:', EPOCHS, '| imágenes de entrenamiento:', ntrain)

RUN = Path('/content/runs/sacred')
saved_last = Path(DRIVE_DIR) / 'last.pt'

def backup(trainer):
    if trainer.epoch % 3 == 0 or trainer.epoch == trainer.epochs - 1:
        for name in ('last.pt', 'best.pt'):
            f = RUN / 'weights' / name
            if f.exists(): shutil.copy(f, Path(DRIVE_DIR) / name)
        if (RUN / 'results.csv').exists(): shutil.copy(RUN / 'results.csv', Path(DRIVE_DIR) / 'results.csv')

if saved_last.exists() and not (RUN / 'weights' / 'last.pt').exists():
    (RUN / 'weights').mkdir(parents=True, exist_ok=True)
    shutil.copy(saved_last, RUN / 'weights' / 'last.pt')

if (RUN / 'weights' / 'last.pt').exists():
    print('Retomo desde last.pt')
    model = YOLO(str(RUN / 'weights' / 'last.pt'))
    model.add_callback('on_fit_epoch_end', backup)
    model.train(resume=True)
else:
    model = YOLO(INIT or BASE_MODEL)
    model.add_callback('on_fit_epoch_end', backup)
    tune = dict(optimizer='AdamW', lr0=0.0005, warmup_epochs=1, patience=10, close_mosaic=8) if INIT else            dict(patience=20, cos_lr=True, close_mosaic=15)
    model.train(data=str(OUT/'data.yaml'), epochs=EPOCHS, imgsz=IMGSZ, batch=16, **tune,
                hsv_h=0.01, hsv_s=0.5, hsv_v=0.4, degrees=15, scale=0.5, translate=0.1, fliplr=0.5, flipud=0.1,
                mosaic=1.0, mixup=0.1, copy_paste=0.3, project='/content/runs', name='sacred', exist_ok=True)

In [ ]:
best = YOLO(str(RUN / 'weights' / 'best.pt'))
m = best.val(data=str(OUT/'data.yaml'), imgsz=IMGSZ, split='val', batch=8, plots=True, project='/content/runs', name='val', exist_ok=True)
print(f"mAP50 máscara: {m.seg.map50:.3f} | mAP50-95 máscara: {m.seg.map:.3f}\n")
for i, c in enumerate(CLASSES):
    try: print(f"  {c:22s} mAP50 {m.seg.class_result(list(m.seg.ap_class_index).index(i))[2]:.3f}")
    except ValueError: print(f"  {c:22s} sin ejemplos en val")

cm = m.confusion_matrix.matrix.astype(int)
names = [c[:10] for c in CLASSES] + ['fondo']
print('\npred \\ real'.ljust(13) + ''.join(n.rjust(11) for n in names))
for i, row in enumerate(cm):
    print(names[i].ljust(13) + ''.join(str(v).rjust(11) for v in row))
for f in glob.glob('/content/runs/val/*.png'):
    shutil.copy(f, DRIVE_DIR)

In [ ]:
# Falsos positivos en fotos sin fruta, de a una imagen para no llenar la GPU.
negs = sorted(glob.glob(str(OUT/'neg_test'/'images'/'*')))
top = []
for j, f in enumerate(negs):
    r = best.predict(f, conf=0.25, imgsz=IMGSZ, verbose=False)[0]
    top.append(float(r.boxes.conf.max()) if len(r.boxes) else 0.0)
    del r
    if j % 50 == 0: torch.cuda.empty_cache()
for conf in [0.25, 0.4, 0.5]:
    fp = sum(t >= conf for t in top)
    print(f"conf {conf}: {fp}/{len(negs)} imágenes sin fruta con alguna detección falsa ({100*fp/len(negs):.1f} %)")

In [ ]:
shutil.copy(RUN / 'weights' / 'best.pt', '/content/best.pt')
shutil.copy(RUN / 'weights' / 'best.pt', Path(DRIVE_DIR) / 'best.pt')
print('best.pt guardado en', DRIVE_DIR)
from google.colab import files
files.download('/content/best.pt')